# TP 2 — Apprentissage supervisé 2A (correction)

**Objectif du sujet.** Étudier un **unique** jeu de données (vins) et y appliquer tous les modèles de
**discrimination** vus en cours : **Naive Bayes**, **analyse discriminante linéaire/quadratique (LDA/QDA)** et
**k plus proches voisins (k-NN)**. On construit un jeu d'apprentissage et un jeu de test, on règle les
hyperparamètres, puis on **compare** les modèles sur le test.

> Correction en **Python / scikit-learn**. Le jeu « wine quality (red) » est téléchargé depuis **OpenML**
> (équivalent du fichier utilisé en cours). On transforme la note en **classe binaire** :
> $Y = 1$ (**bon vin**) si `quality` $\ge 7$ (note de **7 ou 8**), sinon $Y = 0$ (**mauvais vin**, note $\le 6$).

## Brief de démarrage — Les outils de ce TP

> **À lire avant de commencer.** Ce TP compare quatre classifieurs. Voici, en quelques lignes, **comment
> fonctionne chacun** et **ce qu'il faut retenir** avant de coder. Le point commun de tous ces modèles : ils
> cherchent à approcher $\eta(x) = \mathbb{P}(Y = 1 \mid X = x)$ puis décident via la **règle de Bayes** (seuil
> $1/2$). Une illustration graphique de chaque frontière est donnée dans les **annexes** en fin de notebook.

**1. Naive Bayes gaussien — un modèle *génératif*.**
On modélise chaque classe séparément. Par la règle de Bayes,
$\mathbb{P}(Y=c\mid x) \propto \mathbb{P}(Y=c)\prod_j \mathbb{P}(x_j \mid Y=c)$. L'hypothèse **« naïve »** est que
les variables sont **indépendantes conditionnellement à la classe**, et chaque $x_j\mid Y=c$ suit une gaussienne
$\mathcal{N}(\mu_{cj}, \sigma_{cj}^2)$. Concrètement : on ajuste une cloche par variable et par classe, puis on
multiplie. Très **rapide** et **robuste**, mais l'hypothèse d'indépendance est souvent fausse.

**2. Analyse discriminante LDA / QDA — modèles *gaussiens multivariés*.**
On suppose $X \mid Y=c \sim \mathcal{N}(\mu_c, \Sigma_c)$ (une « ellipse » de densité par classe).
- **LDA** : on impose la **même** matrice de covariance à toutes les classes ($\Sigma_c = \Sigma$). La frontière
  de décision est alors une **droite** (un hyperplan).
- **QDA** : chaque classe a **sa propre** covariance. La frontière devient une **courbe** (quadratique), donc
  plus flexible — au prix de beaucoup plus de paramètres à estimer.

**3. k plus proches voisins (k-NN) — modèle *non paramétrique*.**
Aucune hypothèse de forme : pour classer un point, on regarde ses **$k$ voisins** les plus proches et on vote.
$k$ petit ⇒ frontière très découpée (variance forte) ; $k$ grand ⇒ frontière lisse (biais fort). Le k-NN
raisonne par **distances**, il faut donc **standardiser** les variables au préalable.

| Modèle | Hypothèse clé | Frontière | Hyperparamètre | Standardisation |
|---|---|---|---|---|
| Naive Bayes | variables indépendantes / classe | quadratique (diagonale) | `var_smoothing` | inutile |
| LDA | gaussiennes, **covariances égales** | **linéaire** | `shrinkage` | inutile |
| QDA | gaussiennes, covariances libres | **quadratique** | `reg_param` | inutile |
| k-NN | aucune (local) | très flexible | $k$ | **indispensable** |

> **Méthode du TP.** (1) EDA pour comprendre les données ; (2) découpage **stratifié** apprentissage/test ;
> (3) réglage des hyperparamètres par **validation croisée** (on optimise l'**AUC**) ; (4) comparaison finale
> sur le test via **accuracy** et **AUC** (courbes ROC).

In [1]:
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.datasets import fetch_openml
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score, roc_curve
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sys.path += [".", ".."]          # plot_theme.py est a la racine du projet
from plot_theme import *         # couleurs, pick_color(), theme sombre applique par defaut

SEED = 0
np.random.seed(SEED)             # graine unique posee une fois : decoupage, CV et annexes reproductibles

## Partie 1 — Chargement et analyse exploratoire

### Rappel de cours — Pourquoi une analyse exploratoire (EDA) ?

Avant tout modèle, on **regarde les données** pour :
- repérer les **échelles** et **distributions** des variables (utile pour choisir les prétraitements) ;
- détecter un éventuel **déséquilibre des classes** (impacte l'échantillonnage et les métriques) ;
- identifier les variables **discriminantes** (corrélées à la cible) et les **corrélations** entre variables.

Ces observations conditionnent les choix de la Partie 2 (stratification du découpage, standardisation, etc.).


### Question 1 — Chargement et résumé du tableau


In [2]:
wine = fetch_openml("wine-quality-red", version=1, as_frame=True)
X = wine.data.copy()
quality = wine.target.astype(int)          # note de 3 a 8

# Cible binaire : bon vin (note de 7 ou 8) vs mauvais vin (note <= 6)
y = (quality >= 7).astype(int)

print("Dimensions des predicteurs :", X.shape)
print("\nResume statistique :")
X.describe().T

Dimensions des predicteurs : (1599, 11)

Resume statistique :


,count,mean,std,min,25%,50%,75%,max
fixed_acidity,1599.0,8.319637,1.741096,4.60000,7.1000,7.90000,9.200000,15.90000
volatile_acidity,1599.0,0.527821,0.179060,0.12000,0.3900,0.52000,0.640000,1.58000
citric_acid,1599.0,0.270976,0.194801,0.00000,0.0900,0.26000,0.420000,1.00000
residual_sugar,1599.0,2.538806,1.409928,0.90000,1.9000,2.20000,2.600000,15.50000
chlorides,1599.0,0.087467,0.047065,0.01200,0.0700,0.07900,0.090000,0.61100
free_sulfur_dioxide,1599.0,15.874922,10.460157,1.00000,7.0000,14.00000,21.000000,72.00000
total_sulfur_dioxide,1599.0,46.467792,32.895324,6.00000,22.0000,38.00000,62.000000,289.00000
density,1599.0,0.996747,0.001887,0.99007,0.9956,0.99675,0.997835,1.00369
pH,1599.0,3.311113,0.154386,2.74000,3.2100,3.31000,3.400000,4.01000
sulphates,1599.0,0.658149,0.169507,0.33000,0.5500,0.62000,0.730000,2.00000


In [3]:
# Repartition des classes
print("Repartition de la note 'quality' :")
print(quality.value_counts().sort_index())
print("\nRepartition de la classe binaire (0 = mauvais, 1 = bon) :")
print(y.value_counts())
print("\nProportion de bons vins :", round(y.mean(), 3))


Repartition de la note 'quality' :
class
3     10
4     53
5    681
6    638
7    199
8     18
Name: count, dtype: int64

Repartition de la classe binaire (0 = mauvais, 1 = bon) :
class
0    1382
1     217
Name: count, dtype: int64

Proportion de bons vins : 0.136


In [4]:
# Distribution de la variable quality
quality_counts = quality.value_counts().sort_index().rename_axis("quality").reset_index(name="effectif")
fig = px.bar(
    quality_counts, x="quality", y="effectif", text="effectif",
    title="Distribution de la note quality",
    color_discrete_sequence=[COLORWAY[1]]
)
fig.update_traces(textposition="outside")
fig.update_layout(xaxis_title="quality", yaxis_title="Effectif")
apply_dark_theme(fig)
fig.show()


In [5]:
# Distribution de la cible binarisee : 0 = mauvais vin (note <=6), 1 = bon vin (note 7-8)
counts = y.value_counts().sort_index()
bin_df = pd.DataFrame({"y": counts.index.astype(str), "effectif": counts.values})

fig = px.bar(
    bin_df, x="y", y="effectif", color="y", text="effectif",
    title="Distribution de la cible binarisee : 0 = mauvais vin, 1 = bon vin",
    color_discrete_map={"0": CLASS_COLORS["mauvais"], "1": CLASS_COLORS["bon"]},
)
fig.update_traces(textposition="outside")
fig.update_xaxes(type="category")
fig.update_layout(xaxis_title="y (classe binaire)", yaxis_title="Effectif", showlegend=False)
apply_dark_theme(fig)
fig.show()

In [6]:
# Distribution de chaque variable explicative (binning automatique de plotly)
n_cols = 3
n_rows = int(np.ceil(X.shape[1] / n_cols))
fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=list(X.columns),
                    vertical_spacing=0.08, horizontal_spacing=0.08)
for i, var in enumerate(X.columns):
    fig.add_trace(go.Histogram(x=X[var], marker=dict(color=pick_color(i), line=dict(color=PANEL, width=0.4)),
                               opacity=0.88, showlegend=False),
                  row=i // n_cols + 1, col=i % n_cols + 1)
fig.update_annotations(font=dict(size=11))
fig.update_layout(title="Distribution de chaque variable explicative", bargap=0.06, height=300 * n_rows)
fig.show()

**Ce que l'on remarque (réponse Q1) :**
- Les variables sont sur des **échelles très différentes** (ex. `total sulfur dioxide` va jusqu'à ~289,
  `density` ≈ 0.99, `chlorides` ≈ 0.08). ⇒ une **standardisation** sera nécessaire pour le k-NN.
- Plusieurs variables sont **asymétriques** / présentent des **valeurs extrêmes** (résidus sucre, sulfites).
- Avec le seuil « bon vin = note $\ge 7$ », les classes sont **déséquilibrées** : seuls **~14 %** des vins
  sont « bons ». ⇒ on utilisera impérativement un découpage **stratifié** pour préserver cette proportion, et
  on privilégiera l'**AUC** (plutôt que la seule accuracy) pour comparer les modèles (point clé pour la Partie 2).
- Aucune valeur manquante.

### Question 2 — Exploration visuelle : quelles variables influencent la qualité ?

> **La cible est binarisée dès le départ** (`y` = bon/mauvais). On cherche donc la corrélation entre une variable
> **continue** (ex. `alcohol`) et une variable **binaire** (`y`). Le coefficient adapté n'est pas la corrélation
> de Pearson usuelle entre deux variables continues, mais la **corrélation point-bisériale** : c'est le **cas
> particulier de Pearson** obtenu lorsqu'une des deux variables est binaire (codée 0/1). Elle mesure à quel point
> la variable continue **sépare** les deux classes.

On regarde donc la **corrélation point-bisériale** de chaque variable avec la classe, puis la distribution des
variables les plus corrélées selon la classe (**boxplots**).


In [7]:
from scipy.stats import pointbiserialr

# On compare une variable CONTINUE a la cible BINAIRE (0/1) : le coefficient adapte est
# la correlation point-bisereale (cas particulier de Pearson quand une variable est binaire).
corr = X.apply(lambda col: pointbiserialr(y, col)[0]).sort_values()
corr_df = corr.reset_index()
corr_df.columns = ["variable", "correlation"]
corr_df["signe"] = np.where(corr_df["correlation"] >= 0, "positive", "negative")

fig = px.bar(
    corr_df, x="correlation", y="variable", orientation="h", color="signe",
    title="Correlation point-bisereale de chaque variable avec la classe (bon/mauvais)",
    color_discrete_map={"positive": COLORWAY[0], "negative": COLORWAY[1]}
)
fig.update_layout(xaxis_title="Correlation point-bisereale", yaxis_title="")
apply_dark_theme(fig)
fig.show()

corr


volatile_acidity       -0.270712
density                -0.150460
total_sulfur_dioxide   -0.139517
chlorides              -0.097308
free_sulfur_dioxide    -0.071747
pH                     -0.057283
residual_sugar          0.047779
fixed_acidity           0.120061
sulphates               0.199485
citric_acid             0.214716
alcohol                 0.407315
dtype: float64

In [8]:
# Boxplots des variables les plus correlees, selon la classe binaire
top_vars = ["alcohol", "volatile_acidity", "sulphates", "citric_acid"]
data_plot = X.copy()
data_plot["classe"] = y.map({0: "mauvais", 1: "bon"})
long_box = data_plot.melt(id_vars="classe", value_vars=top_vars, var_name="variable", value_name="valeur")

fig = px.box(
    long_box, x="classe", y="valeur", color="classe", facet_col="variable",
    title="Distribution des variables les plus correlees selon la classe",
    color_discrete_map={"mauvais": CLASS_COLORS["mauvais"], "bon": CLASS_COLORS["bon"]}
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_xaxes(title="")
fig.update_yaxes(matches=None)
apply_dark_theme(fig, showlegend=False)
fig.show()


**Ce que l'on remarque (réponse Q2) :** les variables qui semblent **le plus influencer** la qualité sont :
- **`alcohol`** : forte corrélation **positive** (les bons vins sont plus alcoolisés) — variable la plus
  discriminante ;
- **`volatile_acidity`** (acidité volatile) : corrélation **négative** (trop d'acidité volatile = mauvais vin) ;
- **`sulphates`** et **`citric_acid`** : corrélation positive plus modérée.

Ces variables séparent visiblement les deux classes sur les boxplots.


## Partie 2 — Construction des échantillons et entraînement des modèles

### Rappel de cours — Les modèles de discrimination

Tous ces modèles estiment (explicitement ou non) $\eta(x) = \mathbb{P}(Y = 1 \mid X = x)$, puis appliquent la
règle de Bayes (seuil $1/2$).

- **Naive Bayes gaussien.** Modèle **génératif** : par la règle de Bayes,
  $\mathbb{P}(Y=c\mid x) \propto \mathbb{P}(Y=c)\prod_j \mathbb{P}(x_j \mid Y=c)$, avec l'hypothèse
  **« naïve »** d'**indépendance** des variables conditionnellement à la classe, et
  $x_j \mid Y=c \sim \mathcal{N}(\mu_{cj}, \sigma_{cj}^2)$. Rapide, robuste, mais hypothèse forte.
  *Hyperparamètre* : quasi aucun (`var_smoothing`, lissage de variance).

- **Analyse discriminante (LDA / QDA).** On suppose $X \mid Y=c \sim \mathcal{N}(\mu_c, \Sigma_c)$.
  - **LDA** : covariances **égales** ($\Sigma_c = \Sigma$) ⇒ frontière de décision **linéaire**.
  - **QDA** : covariances **différentes** par classe ⇒ frontière **quadratique** (plus flexible, plus de
    paramètres). *Hyperparamètre* : régularisation (`shrinkage` pour LDA, `reg_param` pour QDA).

- **k plus proches voisins.** **Non paramétrique** : vote des $k$ voisins. *Hyperparamètre* : **$k$**.
  Sensible à l'échelle ⇒ **standardiser**.

> **LDA/QDA/Naive Bayes** sont **invariants** à une mise à l'échelle par variable : inutile de standardiser.
> Le **k-NN**, lui, l'exige (via un `Pipeline`).


### Question 1 — Découpage apprentissage / test

Conformément à ce qui a été remarqué en Partie 1 (classes **déséquilibrées**, ~14 % de bons vins), on effectue
un découpage **stratifié** afin de conserver la proportion de bons/mauvais vins dans les deux échantillons.

**Reproductibilité.** La graine est posée **une seule fois** en début de notebook (`np.random.seed(SEED)`) :
aucun `random_state` n'est nécessaire, `train_test_split` puis les modèles tirent leurs nombres aléatoires dans
ce générateur global. En ré-exécutant le notebook **de haut en bas**, on retrouve exactement les mêmes résultats.
Les estimateurs utilisés ici (Naive Bayes, LDA, QDA, k-NN) sont d'ailleurs **déterministes** ; la seule source
d'aléa est le découpage apprentissage / test.

In [9]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y)
print("Apprentissage :", X_tr.shape, "| Test :", X_te.shape)
print("Proportion de bons vins  -> train :", round(y_tr.mean(), 3), "| test :", round(y_te.mean(), 3))

Apprentissage : (1199, 11) | Test : (400, 11)
Proportion de bons vins  -> train : 0.136 | test : 0.135


### Question 2 — Entraînement et réglage des hyperparamètres

On règle les hyperparamètres par **validation croisée** (5 folds) sur l'apprentissage, en optimisant l'**AUC**
(robuste au léger déséquilibre — voir **Annexe H** pour le choix AUC vs accuracy). **Ne pas oublier
d'hyperparamètres !** On règle ici, pour chaque modèle :

- **Naive Bayes** : `var_smoothing` (lissage de variance). *À noter* : la version R (`caret`, méthode `nb`)
  expose **davantage** d'hyperparamètres — `usekernel` (densité par **noyau** au lieu d'une gaussienne),
  `adjust` (largeur de fenêtre du noyau) et `fL` (correction de Laplace). Le principe du noyau est détaillé
  dans l'**Annexe F**. Ici, `np.logspace(-11, -2, 10)` signifie que l'on essaie les valeurs
  $10^{-11}, 10^{-10}, \dots, 10^{-2}$ : les points sont donc **équidistants en échelle logarithmique**.
- **LDA** : `shrinkage` (régularisation de la covariance, solveur `lsqr`) ;
- **QDA** : `reg_param` (régularisation de la covariance par classe) ;
- **k-NN** : `n_neighbors` (= $k$), mais **aussi** `weights` (vote uniforme vs pondéré par la distance) et
  `p` (distance de Manhattan $p=1$ vs euclidienne $p=2$), le tout précédé d'une **standardisation**.

> Le **rôle précis** de chaque hyperparamètre — et les **autres réglages disponibles** pour LDA, QDA et Naive
> Bayes — sont détaillés dans l'**Annexe G**.

> **Pourquoi R et Python ne donnent-ils pas toujours exactement les mêmes résultats ?**

> Les écarts ne viennent pas seulement des hyperparamètres visibles. Ils peuvent aussi venir du **solveur** utilisé
> (par exemple `lsqr` vs `svd`), de la **régularisation** appliquée à la covariance, des conventions de calcul
> numérique, du **découpage** exact des folds en validation croisée, ou encore du traitement des probabilités
> et des égalités en cas de vote. Pour comparer R et Python proprement, il faut donc aligner autant que possible
> les **hyperparamètres**, le **prétraitement**, la **stratégie de validation** et la **graine aléatoire**.


In [10]:
# Un dictionnaire : nom -> (estimateur, grille d'hyperparametres)
var_smoothing_grid = np.logspace(-11, -2, 10)
table_var_smoothing = pd.DataFrame({
    "log10(var_smoothing)": np.log10(var_smoothing_grid),
    "var_smoothing": var_smoothing_grid,
})
print("Valeurs essayees pour var_smoothing :")
display(table_var_smoothing)

modeles = {
    "Naive Bayes": (
        GaussianNB(),
        {"var_smoothing": var_smoothing_grid},
    ),
    "LDA": (
        LinearDiscriminantAnalysis(solver="lsqr"),
        {"shrinkage": np.round(np.linspace(0.0, 0.9, 10), 2)},
    ),
    "QDA": (
        QuadraticDiscriminantAnalysis(),
        {"reg_param": np.round(np.linspace(0.01, 0.9, 10), 3)},
    ),
    "k-NN": (
        Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsClassifier())]),
        {
            "knn__n_neighbors": list(range(1, 41, 2)),
            "knn__weights": ["uniform", "distance"],
            "knn__p": [1, 2],
        },
    ),
}

recherches, meilleurs = {}, {}
for nom, (est, grille) in modeles.items():
    rech = GridSearchCV(est, grille, cv=5, scoring="roc_auc")
    rech.fit(X_tr, y_tr)
    recherches[nom] = rech
    meilleurs[nom] = rech.best_estimator_
    print(f"{nom:12s} | meilleurs params : {rech.best_params_} | AUC CV = {rech.best_score_:.3f}")


Valeurs essayees pour var_smoothing :


,log10(var_smoothing),var_smoothing
0,-11.0,1.000000e-11
1,-10.0,1.000000e-10
2,-9.0,1.000000e-09
3,-8.0,1.000000e-08
4,-7.0,1.000000e-07
5,-6.0,1.000000e-06
6,-5.0,1.000000e-05
7,-4.0,1.000000e-04
8,-3.0,1.000000e-03
9,-2.0,1.000000e-02


Naive Bayes  | meilleurs params : {'var_smoothing': np.float64(1e-10)} | AUC CV = 0.857


LDA          | meilleurs params : {'shrinkage': np.float64(0.0)} | AUC CV = 0.891


QDA          | meilleurs params : {'reg_param': np.float64(0.01)} | AUC CV = 0.853


k-NN         | meilleurs params : {'knn__n_neighbors': 31, 'knn__p': 1, 'knn__weights': 'distance'} | AUC CV = 0.917


**Visualiser l'erreur selon les hyperparamètres.** Pour chaque modèle, on trace d'abord l'**AUC de validation croisée**
en fonction de son hyperparamètre principal. Pour le **k-NN**, la première courbe affiche, pour chaque valeur de $k$,
la **meilleure AUC obtenue** parmi les autres choix possibles (`weights` et `p`) : c'est donc une **enveloppe**
des meilleures performances à $k$ fixé, et non la courbe d'une combinaison unique d'hyperparamètres.

Le deuxième graphe, placé en dessous, désagrège précisément le cas du **k-NN** : il montre l'AUC en fonction de $k$
pour **chaque combinaison** de `weights` et de `p`. On peut ainsi voir si la conclusion "le bon $k$ est grand ou
petit" est robuste, ou si elle dépend fortement de la métrique de distance et du mode de vote.

On y lit le compromis **biais-variance** : pour le k-NN, un $k$ trop petit sur-apprend (frontière trop découpée)
et un $k$ trop grand sous-apprend ; pour la régularisation (LDA/QDA), trop de régularisation lisse à l'excès. Le
pic de la courbe donne la valeur retenue par la validation croisée.


In [11]:
# AUC de validation croisee en fonction de l'hyperparametre principal de chaque modele
axe_hp = {
    "Naive Bayes": ("param_var_smoothing", "var_smoothing", True),
    "LDA": ("param_shrinkage", "shrinkage", False),
    "QDA": ("param_reg_param", "reg_param", False),
    "k-NN": ("param_knn__n_neighbors", "k (n_neighbors)", False),
}

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[
        "Naive Bayes", "LDA",
        "QDA", "k-NN (meilleure AUC pour chaque k)",
        "k-NN selon weights et p", "",
    ],
    vertical_spacing=0.12, horizontal_spacing=0.12,
    specs=[[{}, {}], [{}, {}], [{"colspan": 2}, None]],
)

for i, (nom, (col, titre, logx)) in enumerate(axe_hp.items()):
    res = pd.DataFrame(recherches[nom].cv_results_)
    courbe = res.groupby(col)["mean_test_score"].max().reset_index()
    courbe[col] = courbe[col].astype(float)
    courbe = courbe.sort_values(col)
    r, c = i // 2 + 1, i % 2 + 1
    fig.add_trace(
        go.Scatter(
            x=courbe[col], y=courbe["mean_test_score"], mode="lines+markers",
            line=dict(color=COLORWAY[i % len(COLORWAY)], width=2), showlegend=False,
        ),
        row=r, col=c,
    )
    fig.update_xaxes(title_text=titre, type="log" if logx else "linear", row=r, col=c)
    fig.update_yaxes(title_text="AUC (CV)", row=r, col=c)

res_knn = pd.DataFrame(recherches["k-NN"].cv_results_).copy()
res_knn["param_knn__n_neighbors"] = res_knn["param_knn__n_neighbors"].astype(int)

knn_styles = [
    ("uniform", 1, dict(color=COLORWAY[0], dash="solid")),
    ("uniform", 2, dict(color=COLORWAY[1], dash="solid")),
    ("distance", 1, dict(color=COLORWAY[2], dash="dash")),
    ("distance", 2, dict(color=COLORWAY[3], dash="dash")),
]

for weights, p_value, style in knn_styles:
    courbe = res_knn[
        (res_knn["param_knn__weights"] == weights)
        & (res_knn["param_knn__p"] == p_value)
    ][["param_knn__n_neighbors", "mean_test_score"]].sort_values("param_knn__n_neighbors")
    fig.add_trace(
        go.Scatter(
            x=courbe["param_knn__n_neighbors"],
            y=courbe["mean_test_score"],
            mode="lines+markers",
            name=f"weights={weights}, p={p_value}",
            line=dict(width=2, **style),
        ),
        row=3, col=1,
    )

fig.update_xaxes(title_text="k (n_neighbors)", row=3, col=1)
fig.update_yaxes(title_text="AUC (CV)", row=3, col=1)
fig.update_layout(title="Erreur de validation croisee (AUC) selon les hyperparametres")
fig.update_annotations(font=dict(size=13))
apply_dark_theme(fig, height=1180)
fig.show()

### Question 3 — Comparaison sur le jeu de test

On évalue chaque modèle **une seule fois** sur le test, avec trois métriques :
- l'**accuracy** (taux de bonnes prédictions au seuil $1/2$) ;
- le **recall** de la classe positive, qui mesure la part des **bons vins réellement détectés** ;
- l'**AUC** (aire sous la courbe ROC), qui mesure la capacité à **ordonner** les vins par probabilité d'être
  bons, indépendamment du seuil.


In [12]:
resultats = []
fig = go.Figure()
for idx, (nom, est) in enumerate(meilleurs.items()):
    y_pred = est.predict(X_te)
    y_score = est.predict_proba(X_te)[:, 1]
    acc = accuracy_score(y_te, y_pred)
    rec = recall_score(y_te, y_pred)
    auc = roc_auc_score(y_te, y_score)
    resultats.append({"Modele": nom, "Accuracy": acc, "Recall": rec, "AUC": auc})
    fpr, tpr, _ = roc_curve(y_te, y_score)
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr, mode="lines",
        name=f"{nom} (AUC={auc:.3f})",
        line=dict(color=COLORWAY[idx % len(COLORWAY)], width=3)
    ))

fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines", name="Aleatoire",
    line=dict(color=MUTED, dash="dash")
) )
fig.update_layout(
    title="Courbes ROC sur le jeu de test - comparaison des modeles",
    xaxis_title="Taux de faux positifs",
    yaxis_title="Taux de vrais positifs"
)
apply_dark_theme(fig)
fig.show()

tableau = pd.DataFrame(resultats).sort_values("AUC", ascending=False).reset_index(drop=True)
tableau

,Modele,Accuracy,Recall,AUC
0,k-NN,0.8950,0.500000,0.884634
1,LDA,0.8450,0.425926,0.819578
2,QDA,0.8275,0.314815,0.806840
3,Naive Bayes,0.8325,0.555556,0.804003


### Conclusion — Quel est le modèle le plus prédictif ?

On classe les modèles par **AUC** sur le test (tableau ci-dessus). Les résultats observés sur ce jeu sont nets :

- le **k-NN** est **clairement le meilleur** en AUC (≈ 0.88) et en accuracy (≈ 0.90) : sans hypothèse de forme, il capture
  une frontière **non linéaire** entre bons et mauvais vins, une fois les variables **standardisées** et $k$ réglé
  (ici $k = 31$, distance de Manhattan, vote pondéré par la distance) ;
- la **LDA** arrive **deuxième** (AUC ≈ 0.82). Fait notable : elle **devance la QDA** ici — la plus grande
  flexibilité de la QDA (une covariance **par classe**) ne paie pas, car elle estime beaucoup plus de paramètres
  sur des classes **déséquilibrées**, d'où une **variance d'estimation** plus forte ;
- la **QDA** (AUC ≈ 0.81) et le **Naive Bayes** (AUC ≈ 0.80) ferment la marche du point de vue du classement
  probabiliste. Le Naive Bayes est pénalisé par son hypothèse d'**indépendance** des variables, fausse ici
  (plusieurs variables sont corrélées).

Le **recall** apporte toutefois un autre point de vue : le Naive Bayes récupère ici une part un peu plus grande des
**bons vins** que le k-NN, mais au prix d'un classement global moins bon. Cela illustre qu'en présence de classes
déséquilibrées, **une seule métrique ne suffit pas** : l'AUC mesure la qualité d'ordonnancement, tandis que le recall
mesure la capacité à ne pas manquer les vins réellement bons.

**Le modèle retenu est donc le k-NN** si l'on privilégie l'**AUC** globale. On conserve cependant le recall pour
mieux juger le compromis entre performance globale et détection de la classe minoritaire.

## Annexe A — Pourquoi la règle de Bayes est le point de comparaison central

Le TP2 compare plusieurs classifieurs : Naive Bayes, LDA/QDA et k-NN. Ils ont tous en commun le fait qu’ils cherchent à approcher la fonction
$$\eta(x) = \mathbb{P}(Y=1 \mid X=x).$$

La règle de Bayes
$$g^\star(x)=\mathbb{1}\{\eta(x) \ge 1/2\}$$
est la meilleure décision possible pour le coût 0–1. C’est donc la référence absolue utilisée pour juger la qualité d’un modèle.

### Pourquoi cela aide à interpréter les résultats ?
- Si un modèle a une erreur proche du risque de Bayes, alors il est presque optimal.
- Si son erreur reste nettement plus élevée, cela signifie qu’il reste du bruit, une mauvaise spécification ou un manque d’information.
- Le but n’est pas de « battre » Bayes : c’est de s’en rapprocher autant que possible à partir d’un échantillon fini.

---

## Annexe B — Naive Bayes, LDA/QDA et les frontières de décision

### Naive Bayes : une approximation très utile
Le modèle suppose que les variables sont indépendantes conditionnellement à la classe :
$$\mathbb{P}(X_1,\dots,X_p \mid Y=c) = \prod_j \mathbb{P}(X_j \mid Y=c).$$
Cette hypothèse est souvent fausse, mais elle rend le calcul très simple et souvent robuste.

En pratique, cela revient à dire : pour chaque classe, on estime la densité conditionnelle de chaque variable puis on multiplie les contributions. La règle de décision est alors obtenue en comparant les scores de vraisemblance pour chaque classe.

### LDA / QDA : frontiers linéaires ou quadratiques
Lorsque l’on suppose
$$X \mid Y=c \sim \mathcal{N}(\mu_c, \Sigma_c),$$
les densités conditionnelles sont gaussiennes. On obtient alors une règle de décision qui compare des scores de vraisemblance.

- **LDA** : covariances égales ⇒ frontière de décision **linéaire**.
- **QDA** : covariances différentes ⇒ frontière **quadratique**.

Ainsi, le choix de LDA ou QDA dépend du degré de flexibilité qu’on laisse à la séparation entre classes.

---

## Annexe C — Pourquoi le k-NN exige une standardisation

Le k-NN mesure des distances dans l’espace des variables. Si une variable a une échelle beaucoup plus grande qu’une autre, elle domine la distance et masque les autres informations.

Exemple : si une variable est mesurée en milliers et une autre en unités, la première « écrase » la seconde dans la distance euclidienne. C’est pourquoi on standardise :
$$x_j^{(std)} = \frac{x_j - \mu_j}{\sigma_j}.$$

Cette transformation remet toutes les variables à la même échelle. Elle est cruciale pour les méthodes basées sur les distances, comme le k-NN, et moins nécessaire pour des méthodes comme Naive Bayes ou LDA/QDA.

**Interprétation.** Le k-NN n’est pas seulement un algorithme de vote : il se comporte comme un estimateur local de $\eta(x)$. Le choix de $k$ et l’échelle des variables influencent directement la qualité de cette estimation locale.


---

## Annexe D — Illustration graphique : les frontières de décision

Pour bien **visualiser** ce que font Naive Bayes, LDA et QDA, on les applique à un jeu **synthétique en 2D** :
deux classes gaussiennes ayant des **covariances différentes**. Chaque graphe montre la **probabilité prédite**
$\hat{\mathbb{P}}(Y=1\mid x)$ (fond coloré) et la **frontière de décision** au seuil $1/2$ (ligne blanche).

Ce que l'on doit observer :
- **LDA** : covariances supposées **égales** ⇒ frontière **rigoureusement linéaire** (une droite) ;
- **QDA** : covariances **libres** par classe ⇒ frontière **quadratique** (courbée), qui épouse mieux la forme
  des deux nuages ;
- **Naive Bayes** : gaussiennes à axes **alignés** (indépendance des variables) ⇒ frontière quadratique mais
  contrainte, souvent proche de la QDA quand les variables sont peu corrélées.

In [13]:
# Jeu synthetique 2D : deux classes gaussiennes de covariances differentes
n_pts = 300
X0 = np.random.multivariate_normal([-1.5, -1.0], [[1.3, 0.7], [0.7, 0.6]], n_pts)
X1 = np.random.multivariate_normal([1.6, 1.1],  [[0.7, -0.6], [-0.6, 1.5]], n_pts)
X2d = np.vstack([X0, X1])
y2d = np.r_[np.zeros(n_pts), np.ones(n_pts)].astype(int)

# Grille pour evaluer la probabilite predite
x_min, x_max = X2d[:, 0].min() - 1, X2d[:, 0].max() + 1
y_min, y_max = X2d[:, 1].min() - 1, X2d[:, 1].max() + 1
gx = np.linspace(x_min, x_max, 150)
gy = np.linspace(y_min, y_max, 150)
mx, my = np.meshgrid(gx, gy)
grille = np.column_stack([mx.ravel(), my.ravel()])

modeles_illu = {
    "LDA — frontière linéaire": LinearDiscriminantAnalysis(),
    "QDA — frontière quadratique": QuadraticDiscriminantAnalysis(),
    "Naive Bayes — axes alignés": GaussianNB(),
}

fig = make_subplots(rows=1, cols=3, subplot_titles=list(modeles_illu), horizontal_spacing=0.06)
col0, col1 = pick_class_color(0), pick_class_color(1)

for i, (nom, modele) in enumerate(modeles_illu.items(), start=1):
    modele.fit(X2d, y2d)
    proba = modele.predict_proba(grille)[:, 1].reshape(mx.shape)
    # Fond : probabilite predite d'appartenir a la classe 1
    fig.add_trace(go.Heatmap(
        x=gx, y=gy, z=proba, zmin=0, zmax=1, colorscale=DIVERGING_SCALE,
        opacity=0.75, showscale=(i == 3), colorbar=dict(title="P(Y=1|x)"),
        hoverinfo="skip",
    ), row=1, col=i)
    # Frontiere de decision au seuil 1/2
    fig.add_trace(go.Contour(
        x=gx, y=gy, z=proba, showscale=False, contours=dict(start=0.5, end=0.5, size=1, coloring="none"),
        line=dict(color=TEXT, width=3), hoverinfo="skip",
    ), row=1, col=i)
    # Points d'apprentissage
    for cls, coul, lbl in [(0, col0, "classe 0"), (1, col1, "classe 1")]:
        pts = X2d[y2d == cls]
        fig.add_trace(go.Scatter(
            x=pts[:, 0], y=pts[:, 1], mode="markers", name=lbl, legendgroup=lbl,
            showlegend=(i == 1), marker=dict(color=coul, size=5, line=dict(color=BACKGROUND, width=0.4)),
        ), row=1, col=i)

fig.update_xaxes(title_text="x1")
fig.update_yaxes(title_text="x2", col=1)
fig.update_layout(title="Frontières de décision : LDA (droite) vs QDA (courbe) vs Naive Bayes")
apply_dark_theme(fig, height=430)
fig.show()

---

## Annexe E — Démonstration : pourquoi la frontière est **linéaire** (LDA) ou **quadratique** (QDA)

> **Message central à retenir.** On ne suppose **aucune relation linéaire entre les variables explicatives**.
> L'hypothèse porte uniquement sur la **loi de $X$ dans chaque classe** (gaussienne) et sur le fait que les
> covariances sont **égales** (LDA) ou **libres** (QDA). C'est **la frontière de décision** (dans l'espace des
> $x$) qui hérite de cette forme droite ou courbe — pas une prétendue relation entre les variables.

### Point de départ commun

En classification binaire $Y \in \{0,1\}$, on suppose par classe $X \mid Y=c \sim \mathcal{N}(\hat\mu_c, \hat\Sigma_c)$
et on note $\hat\pi_c$ la proportion de la classe $c$. Par la règle de Bayes :
$$\hat{\mathbb{P}}(Y=c \mid X=x) \;\propto\; \mathcal{N}_x(\hat\mu_c, \hat\Sigma_c)\,\hat\pi_c .$$
La **frontière de décision** est l'ensemble des $x$ tels que $\hat{\mathbb{P}}(Y=1\mid x) = \hat{\mathbb{P}}(Y=0\mid x)$.

### Cas LDA : covariances égales $\hat\Sigma_0 = \hat\Sigma_1 = \hat\Sigma$

En prenant le logarithme de l'égalité $\mathcal{N}_x(\hat\mu_1,\hat\Sigma)\hat\pi_1 = \mathcal{N}_x(\hat\mu_0,\hat\Sigma)\hat\pi_0$ :
$$-\tfrac12 (x-\hat\mu_1)^\top \hat\Sigma^{-1}(x-\hat\mu_1) + \log\hat\pi_1
  = -\tfrac12 (x-\hat\mu_0)^\top \hat\Sigma^{-1}(x-\hat\mu_0) + \log\hat\pi_0 .$$
En développant, le terme quadratique $x^\top \hat\Sigma^{-1} x$ est **identique des deux côtés** et **s'annule** :
$$x^\top \hat\Sigma^{-1}(\hat\mu_1-\hat\mu_0)
  \;-\; \tfrac12 \hat\mu_1^\top \hat\Sigma^{-1}\hat\mu_1
  \;+\; \tfrac12 \hat\mu_0^\top \hat\Sigma^{-1}\hat\mu_0
  \;+\; \log\frac{\hat\pi_1}{\hat\pi_0} \;=\; 0 .$$
C'est une équation de la forme $x^\top w + b = 0$ : **l'équation d'un hyperplan**. La frontière est donc
**linéaire** — d'où le nom « analyse discriminante *linéaire* ».

### Cas QDA : covariances libres $\hat\Sigma_0 \neq \hat\Sigma_1$

Le même calcul, sans pouvoir simplifier les covariances, donne :
$$-\tfrac12\, x^\top (\hat\Sigma_1^{-1} - \hat\Sigma_0^{-1})\, x
  \;+\; x^\top (\hat\Sigma_1^{-1}\hat\mu_1 - \hat\Sigma_0^{-1}\hat\mu_0)
  \;-\; \tfrac12 \hat\mu_1^\top \hat\Sigma_1^{-1}\hat\mu_1
  \;+\; \tfrac12 \hat\mu_0^\top \hat\Sigma_0^{-1}\hat\mu_0
  \;+\; \log\frac{\hat\pi_1}{\hat\pi_0}
  \;-\; \tfrac12 \log\frac{\det\hat\Sigma_1}{\det\hat\Sigma_0} \;=\; 0 .$$
C'est cette fois de la forme $x^\top A x + x^\top w + b = 0$ : **l'équation d'une surface quadratique**. La
frontière est donc **quadratique** (courbée) — d'où « analyse discriminante *quadratique* ».

### La conclusion clé

Le **terme quadratique** $-\tfrac12\, x^\top (\hat\Sigma_1^{-1} - \hat\Sigma_0^{-1})\, x$ **disparaît exactement
lorsque $\hat\Sigma_0 = \hat\Sigma_1$** : la QDA **redonne** alors la LDA. Autrement dit, la forme de la
frontière (droite ou courbe) est une **conséquence mathématique** de l'hypothèse faite sur les covariances,
et non une hypothèse de linéarité entre les variables.


---

## Annexe F — Le principe du **noyau** (kernel)

### À quoi ça sert ?

Le Naive Bayes gaussien suppose que chaque variable suit, **dans une classe**, une **gaussienne**
$\mathcal{N}(\mu_{cj}, \sigma_{cj}^2)$. Mais une variable réelle peut être **asymétrique** ou **bimodale** :
une seule cloche gaussienne la représente alors mal. L'**estimation par noyau** (*kernel density estimation*,
KDE) remplace cette hypothèse paramétrique par une estimation **non paramétrique** de la densité :
$$\hat f_h(x) \;=\; \frac{1}{n\,h}\sum_{i=1}^{n} K\!\left(\frac{x - x_i}{h}\right),$$
où $K$ est un **noyau** (fonction positive, symétrique, d'intégrale 1) et $h > 0$ la **fenêtre** (*bandwidth*).
On dépose une petite bosse $K$ sur chaque observation, puis on les somme.

### À quoi ça ressemble ?

Noyaux courants (vus en cours) :
$$K_{\text{gaussien}}(t) = \tfrac{1}{\sqrt{2\pi}} e^{-t^2/2}, \qquad
  K_{\text{Epanechnikov}}(t) = \tfrac{3}{4}(1 - t^2)\,\mathbb{1}_{|t|\le 1}.$$
Le rôle de la **fenêtre $h$** est exactement celui du $k$ des plus proches voisins : $h$ **petit** ⇒ on lisse peu
(estimation « bruitée », variance forte) ; $h$ **grand** ⇒ on lisse beaucoup (estimation trop plate, biais fort).
On choisit $h$ par **validation croisée**.

### Le lien avec les hyperparamètres du Naive Bayes en R

C'est précisément ce que règlent les hyperparamètres de la méthode `nb` de `caret` :
- **`usekernel`** : utiliser une densité par **noyau** (KDE) au lieu de la gaussienne ;
- **`adjust`** : un multiplicateur de la **fenêtre $h$** (plus ou moins de lissage) ;
- **`fL`** : la **correction de Laplace** (pour les variables discrètes, évite les probabilités nulles).

### Quel intérêt en pratique ?

Quand la vraie densité conditionnelle est **loin d'une gaussienne**, la KDE l'estime bien mieux ⇒ un Naive Bayes
« à noyau » est **plus fidèle** et souvent **plus prédictif**. Le prix à payer : un hyperparamètre de plus ($h$)
à régler, et un léger surcoût de calcul. L'illustration ci-dessous compare, pour une variable de vin, l'ajustement
**gaussien** et l'estimation **par noyau**, puis montre l'effet de la fenêtre $h$.


In [14]:
from scipy.stats import gaussian_kde, norm

# Exemple concret : densite de 'alcohol' chez les bons vins, estimee de deux facons
var = "alcohol"
vals = X.loc[y == 1, var].to_numpy()
g = np.linspace(vals.min(), vals.max(), 200)

fig = make_subplots(
    rows=1, cols=2, horizontal_spacing=0.1,
    subplot_titles=["Gaussienne vs noyau (usekernel)", "Effet de la fenetre h (adjust)"],
)

# Panneau 1 : hypothese gaussienne vs estimation par noyau
fig.add_trace(go.Histogram(x=vals, histnorm="probability density", nbinsx=30,
              name="histogramme", marker=dict(color=MUTED), opacity=0.4), row=1, col=1)
fig.add_trace(go.Scatter(x=g, y=norm.pdf(g, vals.mean(), vals.std()), mode="lines",
              name="hypothese gaussienne", line=dict(color=COLORWAY[1], width=3)), row=1, col=1)
fig.add_trace(go.Scatter(x=g, y=gaussian_kde(vals)(g), mode="lines",
              name="noyau (KDE)", line=dict(color=COLORWAY[0], width=3)), row=1, col=1)

# Panneau 2 : effet de la fenetre h (hyperparametre 'adjust')
base = gaussian_kde(vals).factor
for j, (mult, lbl) in enumerate([(0.3, "h petit"), (1.0, "h defaut"), (2.5, "h grand")]):
    dens = gaussian_kde(vals, bw_method=base * mult)(g)
    fig.add_trace(go.Scatter(x=g, y=dens, mode="lines", name=lbl,
                  line=dict(color=COLORWAY[j], width=2)), row=1, col=2)

fig.update_xaxes(title_text=var)
fig.update_yaxes(title_text="densite", col=1)
fig.update_layout(title=f"Estimation par noyau de la densite de '{var}' (bons vins)")
apply_dark_theme(fig, height=420)
fig.show()

---

## Annexe G — Les hyperparamètres de chaque modèle : rôle et autres réglages possibles

Un **hyperparamètre** n'est pas appris sur les données : on le **fixe avant** l'entraînement (ici par validation
croisée). Voici, modèle par modèle, **à quoi correspond** chaque hyperparamètre et **quels autres** on pourrait
régler.

### Ce que fait la régularisation, en pratique

Sur un échantillon fini, la covariance ou la variance empirique peut être **bruitée** : certaines corrélations
sont dues au hasard, et l'inversion de la matrice de covariance peut devenir instable quand les variables sont
nombreuses ou redondantes. **Régulariser** consiste à remplacer l'estimateur brut par une version un peu plus
simple et plus stable. On accepte alors un peu plus de **biais** pour réduire la **variance d'estimation**.

Important : on ne modifie pas les données elles-mêmes. On modifie l'**estimateur** des variances / covariances
utilisé par le modèle. En général, la régularisation ne change donc pas les valeurs observées de `X`, mais elle
change la façon dont le modèle pondère les directions de l'espace.

### Que fait le shrinkage ?

Dans `scikit-learn`, le **shrinkage** de la LDA remplace la covariance empirique $\hat\Sigma$ par une version
plus stable de la forme
$$\hat\Sigma_{\alpha} = (1-\alpha)\hat\Sigma + \alpha\,\mu I, \qquad \mu = \frac{\mathrm{tr}(\hat\Sigma)}{p}.$$
Autrement dit, on tire la covariance vers une **identité mise à l'échelle** : les variances extrêmes et les
corrélations trop marquées sont atténuées. Plus $\alpha$ est grand, plus la matrice devient proche d'une forme
"ronde" et moins le modèle fait confiance aux corrélations fines estimées dans l'échantillon.

- En **LDA**, `shrinkage` régularise donc la **covariance commune** : `0` correspond à la covariance empirique
  brute ; des valeurs plus grandes simplifient progressivement la matrice et la rendent plus stable.
- En **QDA**, `reg_param` joue un rôle voisin **classe par classe** :
  $$\hat\Sigma_{c,\gamma} = (1-\gamma)\hat\Sigma_c + \gamma I.$$
  On tire chaque covariance de classe vers l'identité, ce qui évite des inversions instables et des frontières
  trop tortueuses.

### Naive Bayes gaussien (`GaussianNB`)
- **`var_smoothing`** *(réglé ici)* : `GaussianNB` remplace chaque variance estimée $\hat\sigma_{cj}^2$ par
  $$\tilde\sigma_{cj}^2 = \hat\sigma_{cj}^2 + \varepsilon \max_{k,\ell}(\hat\sigma_{k\ell}^2),$$
  où $\varepsilon = \texttt{var\_smoothing}`. On ajoute donc le **même petit plancher** à toutes les variances.
  Cela **stabilise** le calcul en évitant des variances quasi nulles. Plus `var_smoothing` est grand, plus les
  cloches gaussiennes sont **élargies**.
- *Autre réglage* : **`priors`** (fixer les probabilités a priori $\mathbb{P}(Y=c)$ au lieu de les estimer).
- En pratique, `GaussianNB` a **très peu** d'hyperparamètres. Pour en avoir plus (densité par **noyau**), il faut
  une autre implémentation — c'est ce que fait la version **R** (`klaR` : `usekernel`/`adjust`/`fL`, cf Annexe F).

### LDA (`LinearDiscriminantAnalysis`)
- **`shrinkage`** *(réglé ici)* : **régularise** la matrice de covariance commune en la tirant vers une identité
  mise à l'échelle. Utile quand il y a **peu d'observations** pour le nombre de variables. Nécessite le solveur
  `lsqr` ou `eigen` (pas `svd`).
- *Autres réglages* : **`solver`** (`svd`, `lsqr`, `eigen`), **`priors`**, et **`n_components`** (uniquement pour
  la **réduction de dimension**, pas pour la classification ici).

### QDA (`QuadraticDiscriminantAnalysis`)
- **`reg_param`** *(réglé ici)* : **régularise** la covariance **de chaque classe** en la tirant vers l'identité
  ($0$ = aucune régularisation). C'est **important** car la QDA estime **une covariance par classe** (beaucoup de
  paramètres) : sans régularisation, ces matrices peuvent être **mal conditionnées / singulières**.
- *Autre réglage* : **`priors`**, **`tol`** (seuil numérique).

### k plus proches voisins (`KNeighborsClassifier`)
- **`n_neighbors`** ($k$) *(réglé ici)* : nombre de voisins qui votent. Petit $k$ ⇒ **sur-apprentissage** ;
  grand $k$ ⇒ **sous-apprentissage**.
- **`weights`** *(réglé ici)* : `uniform` (tous les voisins comptent pareil) vs `distance` (les voisins proches
  **pèsent plus**).
- **`p`** *(réglé ici)* : paramètre de la distance de Minkowski — $p=1$ **Manhattan**, $p=2$ **euclidienne**.
- *Autre réglage* : **`metric`** (autres distances) et **`algorithm`** (`kd_tree`/`ball_tree`/`brute`, qui
  n'influence que la **vitesse**, pas les résultats).

> **À retenir.** LDA, QDA et Naive Bayes ont **peu** d'hyperparamètres (essentiellement une **régularisation**),
> parce que l'essentiel de leur comportement est fixé par leurs **hypothèses de modèle**. Le k-NN, non
> paramétrique, se règle surtout via $k$ (et la façon de pondérer/mesurer les distances).


### Visualisations — Schémas explicatifs des hyperparamètres

Les graphiques ci-dessous sont construits sur des **exemples jouets** pour isoler un seul effet à la fois.

- Pour la **LDA**, on regarde comment `shrinkage` rend la **covariance commune** plus ronde.
- Pour le **Naive Bayes gaussien**, on regarde comment `var_smoothing` **élargit** les Gaussiennes de classe.
- Pour la **QDA**, on regarde comment `reg_param` rend les **covariances de classe** plus isotropes et la
  frontière de décision plus lisse.

L'idée n'est pas ici de comparer les performances, mais de voir **ce que l'hyperparamètre change dans la forme**
du modèle : ellipse de covariance pour la LDA, largeur des densités pour le Naive Bayes, et forme des
ellipses / frontière pour la QDA.

In [15]:
# Schémas explicatifs : LDA (covariance), Naive Bayes (variance), QDA (frontière)

def ellipse_points(mean, cov, scale=2.0, n=240):
    vals, vecs = np.linalg.eigh(cov)
    order = np.argsort(vals)[::-1]
    vals = np.maximum(vals[order], 1e-12)
    vecs = vecs[:, order]
    theta = np.linspace(0, 2 * np.pi, n)
    circle = np.vstack([np.cos(theta), np.sin(theta)])
    ellipse = mean.reshape(2, 1) + vecs @ np.diag(np.sqrt(vals) * scale) @ circle
    return ellipse[0], ellipse[1]

col0 = pick_class_color(0)
col1 = pick_class_color(1)

# LDA : une covariance commune qui se rapproche d'une forme isotrope
mean_lda_0 = np.array([-1.7, -1.0])
mean_lda_1 = np.array([1.6, 1.3])
cov_lda = np.array([[1.6, 1.05], [1.05, 0.9]])
X0_lda = np.random.multivariate_normal(mean_lda_0, cov_lda, 120)
X1_lda = np.random.multivariate_normal(mean_lda_1, cov_lda, 120)
X_lda = np.vstack([X0_lda, X1_lda])
y_lda = np.r_[np.zeros(len(X0_lda)), np.ones(len(X1_lda))].astype(int)
cov_emp = np.cov(X_lda, rowvar=False, bias=True)
mu_cov = np.trace(cov_emp) / cov_emp.shape[0]
identity_scaled = mu_cov * np.eye(cov_emp.shape[0])
shrinkage_values = [0.0, 0.4, 0.8]
mean_lda = X_lda.mean(axis=0)

# Naive Bayes : smoothing sur une 1D gaussienne pour voir la largeur changer
nb_samples = {
    0: np.random.normal(-1.3, 0.28, 120),
    1: np.random.normal(1.4, 0.45, 120),
}
nb_means = {cls: values.mean() for cls, values in nb_samples.items()}
nb_vars = {cls: values.var(ddof=0) for cls, values in nb_samples.items()}
nb_max_var = max(nb_vars.values())
var_smoothing_values = [0.0, 0.3, 1.0]
x_grid = np.linspace(-3.5, 3.5, 400)

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        "LDA shrinkage = 0.0", "LDA shrinkage = 0.4", "LDA shrinkage = 0.8",
        "Naive Bayes var_smoothing = 0.0", "Naive Bayes var_smoothing = 0.3", "Naive Bayes var_smoothing = 1.0",
    ],
    vertical_spacing=0.15, horizontal_spacing=0.08,
    specs=[[{}, {}, {}], [{}, {}, {}]],
)

for col_idx, alpha in enumerate(shrinkage_values, start=1):
    cov_shrunk = (1 - alpha) * cov_emp + alpha * identity_scaled
    ex, ey = ellipse_points(mean_lda, cov_shrunk, scale=2.0)
    fig.add_trace(
        go.Scatter(
            x=X0_lda[:, 0], y=X0_lda[:, 1], mode="markers",
            name="classe 0", legendgroup="lda-classe-0", showlegend=(col_idx == 1),
            marker=dict(color=col0, size=5, opacity=0.55),
        ),
        row=1, col=col_idx,
    )
    fig.add_trace(
        go.Scatter(
            x=X1_lda[:, 0], y=X1_lda[:, 1], mode="markers",
            name="classe 1", legendgroup="lda-classe-1", showlegend=(col_idx == 1),
            marker=dict(color=col1, size=5, opacity=0.55),
        ),
        row=1, col=col_idx,
    )
    fig.add_trace(
        go.Scatter(
            x=ex, y=ey, mode="lines",
            name="ellipse de covariance", legendgroup="lda-ellipse", showlegend=(col_idx == 1),
            line=dict(color=TEXT, width=3),
        ),
        row=1, col=col_idx,
    )
    fig.add_trace(
        go.Scatter(
            x=[mean_lda[0]], y=[mean_lda[1]], mode="markers",
            name="centre", legendgroup="lda-centre", showlegend=(col_idx == 1),
            marker=dict(color=TEXT, size=10, symbol="x"),
        ),
        row=1, col=col_idx,
    )

for col_idx, eps in enumerate(var_smoothing_values, start=1):
    for cls, color, label in [(0, col0, "classe 0"), (1, col1, "classe 1")]:
        variance = nb_vars[cls] + eps * nb_max_var
        density = 1 / np.sqrt(2 * np.pi * variance) * np.exp(-0.5 * ((x_grid - nb_means[cls]) ** 2) / variance)
        fig.add_trace(
            go.Scatter(
                x=x_grid, y=density, mode="lines", fill="tozeroy",
                name=label, legendgroup=f"nb-{cls}", showlegend=(col_idx == 1),
                line=dict(color=color, width=3), opacity=0.9,
            ),
            row=2, col=col_idx,
        )
        fig.add_trace(
            go.Scatter(
                x=nb_samples[cls], y=np.full_like(nb_samples[cls], -0.02 * (cls + 1)), mode="markers",
                name=f"points {label}", legendgroup=f"nb-points-{cls}", showlegend=False,
                marker=dict(color=color, size=5, opacity=0.6),
            ),
            row=2, col=col_idx,
        )

for col_idx in range(1, 4):
    fig.update_xaxes(title_text="x1", row=1, col=col_idx)
    fig.update_yaxes(title_text="x2", row=1, col=col_idx)
    fig.update_xaxes(title_text="variable 1D", row=2, col=col_idx)
    fig.update_yaxes(title_text="densite", row=2, col=col_idx)

fig.update_layout(title="LDA, Naive Bayes et leurs hyperparametres : schema geometrique")
apply_dark_theme(fig, height=780)
fig.show()

In [16]:
# Visualiser l'effet de reg_param sur les ellipses de covariance et la frontiere de la QDA
n_pts_reg = 250
X0_reg = np.random.multivariate_normal([-1.4, -1.1], [[1.5, 0.9], [0.9, 0.8]], n_pts_reg)
X1_reg = np.random.multivariate_normal([1.5, 1.0], [[0.8, -0.7], [-0.7, 1.7]], n_pts_reg)
X_reg = np.vstack([X0_reg, X1_reg])
y_reg = np.r_[np.zeros(n_pts_reg), np.ones(n_pts_reg)].astype(int)

x_min, x_max = X_reg[:, 0].min() - 1, X_reg[:, 0].max() + 1
y_min, y_max = X_reg[:, 1].min() - 1, X_reg[:, 1].max() + 1
gx = np.linspace(x_min, x_max, 150)
gy = np.linspace(y_min, y_max, 150)
mx, my = np.meshgrid(gx, gy)
grid_reg = np.column_stack([mx.ravel(), my.ravel()])

reg_values = [0.0, 0.2, 0.8]
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f"QDA reg_param = {value}" for value in reg_values],
    horizontal_spacing=0.06,
)

col0 = pick_class_color(0)
col1 = pick_class_color(1)

for idx, reg_value in enumerate(reg_values, start=1):
    qda_reg = QuadraticDiscriminantAnalysis(reg_param=reg_value)
    qda_reg.fit(X_reg, y_reg)
    proba = qda_reg.predict_proba(grid_reg)[:, 1].reshape(mx.shape)
    covariances = []
    for cls_idx in range(2):
        rotations = qda_reg.rotations_[cls_idx]
        scalings = np.maximum(qda_reg.scalings_[cls_idx], 1e-12)
        covariances.append(rotations @ np.diag(scalings) @ rotations.T)

    fig.add_trace(
        go.Heatmap(
            x=gx, y=gy, z=proba, zmin=0, zmax=1,
            colorscale=DIVERGING_SCALE, opacity=0.75,
            showscale=(idx == 3), colorbar=dict(title="P(Y=1|x)"), hoverinfo="skip",
        ),
        row=1, col=idx,
    )
    fig.add_trace(
        go.Contour(
            x=gx, y=gy, z=proba, showscale=False,
            contours=dict(start=0.5, end=0.5, size=1, coloring="none"),
            line=dict(color=TEXT, width=3), hoverinfo="skip",
        ),
        row=1, col=idx,
    )
    for cls, color, label in [(0, col0, "classe 0"), (1, col1, "classe 1")]:
        points = X_reg[y_reg == cls]
        fig.add_trace(
            go.Scatter(
                x=points[:, 0], y=points[:, 1], mode="markers",
                name=label, legendgroup=label, showlegend=(idx == 1),
                marker=dict(color=color, size=5, line=dict(color=BACKGROUND, width=0.4)),
            ),
            row=1, col=idx,
        )
        ex, ey = ellipse_points(qda_reg.means_[cls], covariances[cls], scale=2.0)
        fig.add_trace(
            go.Scatter(
                x=ex, y=ey, mode="lines",
                name=f"ellipse {label}", legendgroup=f"ellipse-{label}", showlegend=(idx == 1),
                line=dict(color=color, width=3, dash="dash"),
                opacity=0.95,
            ),
            row=1, col=idx,
        )

fig.update_xaxes(title_text="x1")
fig.update_yaxes(title_text="x2", col=1)
fig.update_layout(title="Effet de reg_param sur la QDA : ellipses et frontiere")
apply_dark_theme(fig, height=430)
fig.show()

---

## Annexe H — Pourquoi optimiser l'**AUC** plutôt que l'**accuracy** ?

On utilise l'**AUC** (aire sous la courbe ROC) — et non l'accuracy — aussi bien pour **régler les
hyperparamètres** (validation croisée) que pour **comparer** les modèles sur le test. Trois raisons.

**1. Les classes sont déséquilibrées.** Seuls ~14 % des vins sont « bons ». Un classifieur trivial qui prédirait
**toujours « mauvais »** obtiendrait déjà ~86 % d'accuracy **sans rien apprendre**. L'accuracy récompense donc la
classe majoritaire et **masque** l'incapacité à repérer les bons vins. L'AUC, elle, est **insensible** à la
proportion des classes.

**2. L'accuracy dépend d'un seuil arbitraire.** Elle fige la décision au **seuil $1/2$**. Or un bon modèle peut
très bien **ordonner** les vins (scores élevés pour les bons) tout en ayant un seuil optimal différent de $1/2$.
L'AUC évalue la qualité de cet **ordonnancement** pour **tous les seuils** à la fois.

**3. Interprétation probabiliste claire.** L'AUC est **la probabilité qu'un bon vin tiré au hasard reçoive un
score plus élevé qu'un mauvais vin tiré au hasard** :
$$\text{AUC} = \mathbb{P}\big(\hat{s}(X^{+}) > \hat{s}(X^{-})\big), \qquad X^{+}\ \text{bon},\ X^{-}\ \text{mauvais}.$$
Une AUC de $0.5$ correspond au hasard, $1.0$ à un classement parfait.

**Pour le réglage des hyperparamètres**, optimiser l'AUC évite de choisir des réglages qui se contentent de
prédire la classe majoritaire. **Pour la comparaison finale**, elle offre une mesure **équitable** et
**indépendante du seuil**. On conserve tout de même l'**accuracy** comme métrique **complémentaire** et
facilement interprétable.
